# Participant-Level Data Splitting Strategy

This notebook implements the participant-level data splitting strategy for the PADS movement metadata dataset. The objective is to prepare the dataset for machine learning by dividing participants into training, validation, and testing sets while preventing participant-level data leakage.

The splitting strategy ensures that all recordings belonging to the same participant remain within a single dataset, allowing unbiased model training and evaluation.

In [1]:
# ============================================================
# Import Libraries
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

# Set random seed
np.random.seed(42)

In [2]:
# ============================================================
# Load  Datasets
# ============================================================

patients_df = pd.read_csv("../data/interim/patients.csv")
movement_df = pd.read_csv("../data/interim/movement_metadata.csv")

print(f"Patients: {len(patients_df)}")
print(f"Movement records: {len(movement_df)}")

display(patients_df.head())
display(movement_df.head())

Patients: 469
Movement records: 5159


,patient_id,study_id,condition,label,disease_comment,age_at_diagnosis,age,height_cm,weight_kg,gender,handedness,appearance_in_kinship,appearance_in_first_grade_kinship,effect_of_alcohol_on_tremor
0,1,PADS,Healthy,0,-,56,56,173,78,male,right,True,True,Unknown
1,2,PADS,Other Movement Disorders,2,Left-Sided resting tremor and hypokinesia with...,69,81,193,104,male,right,False,NaN,No effect
2,3,PADS,Healthy,0,-,45,45,170,78,female,right,False,NaN,Unknown
3,4,PADS,Parkinson's,1,IPS akinetic-rigid type,63,67,161,90,female,right,False,NaN,No effect
4,5,PADS,Parkinson's,1,IPS tremordominant type,65,75,172,86,male,left,False,NaN,Unknown


,patient_id,device,sampling_rate,task,samples,left_file,right_file
0,1,Apple Watch Series 4,100,CrossArms,1024,timeseries/001_CrossArms_LeftWrist.txt,timeseries/001_CrossArms_RightWrist.txt
1,1,Apple Watch Series 4,100,DrinkGlas,1024,timeseries/001_DrinkGlas_LeftWrist.txt,timeseries/001_DrinkGlas_RightWrist.txt
2,1,Apple Watch Series 4,100,Entrainment,2048,timeseries/001_Entrainment_LeftWrist.txt,timeseries/001_Entrainment_RightWrist.txt
3,1,Apple Watch Series 4,100,HoldWeight,1024,timeseries/001_HoldWeight_LeftWrist.txt,timeseries/001_HoldWeight_RightWrist.txt
4,1,Apple Watch Series 4,100,LiftHold,1024,timeseries/001_LiftHold_LeftWrist.txt,timeseries/001_LiftHold_RightWrist.txt


In [3]:
# =============================================================================
# Dataset Summary
# =============================================================================

print(f"Movement recordings : {len(movement_df):,}")
print(f"Participants        : {movement_df['patient_id'].nunique():,}")
print(f"Motor tasks         : {movement_df['task'].nunique():,}")

Movement recordings : 5,159
Participants        : 469
Motor tasks         : 11


In [4]:
# =============================================================================
# Dataset Structure
# =============================================================================

movement_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5159 entries, 0 to 5158
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   patient_id     5159 non-null   int64
 1   device         5159 non-null   str  
 2   sampling_rate  5159 non-null   int64
 3   task           5159 non-null   str  
 4   samples        5159 non-null   int64
 5   left_file      5159 non-null   str  
 6   right_file     5159 non-null   str  
dtypes: int64(3), str(4)
memory usage: 282.3 KB


In [5]:
# =============================================================================
# Check Unique Participants
# =============================================================================

movement_df["patient_id"].nunique()

469

In [6]:
# ============================================================
# Split Patients into Training and Temporary Sets 
# ============================================================

train_patients, temp_patients = train_test_split(
    patients_df,
    test_size=0.30,
    stratify=patients_df["label"],
    random_state=42,
)

train_patients = train_patients.reset_index(drop=True)
temp_patients = temp_patients.reset_index(drop=True)

In [7]:
# ============================================================
# Verify First Split
# ============================================================

print(f"Training patients : {len(train_patients):,}")
print(f"Temporary patients: {len(temp_patients):,}")

print()

print(f"Unique training patients : {train_patients['patient_id'].nunique()}")
print(f"Unique temporary patients: {temp_patients['patient_id'].nunique()}")

Training patients : 328
Temporary patients: 141

Unique training patients : 328
Unique temporary patients: 141


In [8]:
# ============================================================
# Split Temporary Patients into Validation and Test Sets 
# ============================================================

validation_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.50,
    stratify=temp_patients["label"],
    random_state=42,
)

validation_patients = validation_patients.reset_index(drop=True)
test_patients = test_patients.reset_index(drop=True)

In [9]:
# ============================================================
# Check HC / PD / OMD Distribution
# ============================================================

def map_group(condition):
    if condition == "Healthy":
        return "HC"
    elif condition == "Parkinson's":
        return "PD"
    else:
        return "OMD"

for name, df in [
    ("Training", train_patients),
    ("Validation", validation_patients),
    ("Test", test_patients),
]:

    temp = df.copy()
    temp["Group"] = temp["condition"].apply(map_group)

    counts = temp["Group"].value_counts().reindex(["HC", "PD", "OMD"], fill_value=0)
    percentages = (
        temp["Group"]
        .value_counts(normalize=True)
        .reindex(["HC", "PD", "OMD"], fill_value=0)
        * 100
    ).round(2)

    print(f"\n{name} Set")
    print("=" * 40)

    summary = pd.DataFrame({
        "Count": counts,
        "Percentage (%)": percentages
    })

    print(summary)


Training Set
       Count  Percentage (%)
Group                       
HC        55           16.77
PD       193           58.84
OMD       80           24.39

Validation Set
       Count  Percentage (%)
Group                       
HC        12           17.14
PD        41           58.57
OMD       17           24.29

Test Set
       Count  Percentage (%)
Group                       
HC        12           16.90
PD        42           59.15
OMD       17           23.94


## Class Distribution Summary

The HC, PD, and OMD groups are represented across the training, validation, and test datasets. Although minor differences exist due to the relatively small validation and test sets, the overall class distributions remain reasonably consistent across all three datasets, indicating that the stratified participant-level splitting strategy successfully preserved the diagnostic group proportions.

In [10]:
# =============================================================================
# Check for Data Leakage
# =============================================================================

train_ids = set(train_patients["patient_id"])
validation_ids = set(validation_patients["patient_id"])
test_ids = set(test_patients["patient_id"])

print("Train ∩ Validation:", len(train_ids & validation_ids))
print("Train ∩ Test      :", len(train_ids & test_ids))
print("Validation ∩ Test :", len(validation_ids & test_ids))

Train ∩ Validation: 0
Train ∩ Test      : 0
Validation ∩ Test : 0


## Data Leakage Summary

A participant-level stratified data splitting strategy was successfully implemented using `train_test_split()` with stratification based on participant labels. The movement metadata dataset was divided into training, validation, and testing datasets while ensuring that every participant was assigned to only one dataset.

Verification confirmed that there was no overlap of participant IDs between the training, validation, and test datasets, eliminating participant-level data leakage. The resulting datasets are now ready for feature engineering, model development, and performance evaluation.

In [11]:
# ============================================================
# Create Participant Split Table
# ============================================================

train_split = train_patients[["patient_id", "label"]].copy()
train_split["split"] = "train"

validation_split = validation_patients[["patient_id", "label"]].copy()
validation_split["split"] = "validation"

test_split = test_patients[["patient_id", "label"]].copy()
test_split["split"] = "test"

participant_split = pd.concat(
    [train_split, validation_split, test_split],
    ignore_index=True
)

participant_split = participant_split.sort_values(
    by="patient_id"
).reset_index(drop=True)

display(participant_split.head())


print("\nParticipants per split:")
print(participant_split["split"].value_counts())

print(f"\nTotal participants: {len(participant_split)}")
print(f"Unique participants: {participant_split['patient_id'].nunique()}")

,patient_id,label,split
0,1,0,train
1,2,2,train
2,3,0,validation
3,4,1,validation
4,5,1,train



Participants per split:
split
train         328
test           71
validation     70
Name: count, dtype: int64

Total participants: 469
Unique participants: 469


In [12]:
# ============================================================
# Apply Participant Split to Movement Dataset
# ============================================================

movement_df = movement_df.merge(
    participant_split[["patient_id", "split"]],
    on="patient_id",
    how="left"
)

print("Missing split:", movement_df["split"].isna().sum())

train_df = movement_df[
    movement_df["split"] == "train"
].reset_index(drop=True)

validation_df = movement_df[
    movement_df["split"] == "validation"
].reset_index(drop=True)

test_df = movement_df[
    movement_df["split"] == "test"
].reset_index(drop=True)

Missing split: 0


In [13]:
# =============================================================================
# Verify Final Split
# =============================================================================

print(f"Training participants   : {train_df['patient_id'].nunique()}")
print(f"Validation participants : {validation_df['patient_id'].nunique()}")
print(f"Test participants       : {test_df['patient_id'].nunique()}")

print()

print(f"Training recordings   : {len(train_df):,}")
print(f"Validation recordings : {len(validation_df):,}")
print(f"Test recordings       : {len(test_df):,}")

Training participants   : 328
Validation participants : 70
Test participants       : 71

Training recordings   : 3,608
Validation recordings : 770
Test recordings       : 781


In [14]:
# ============================================================
# Save Split Datasets
# ============================================================

OUTPUT_PATH = Path("../data/processed")

# Save participant split table
participant_split.to_csv(
    OUTPUT_PATH / "participant_split.csv",
    index=False
)

# Save movement metadata splits
train_df.to_csv(
    OUTPUT_PATH / "train_metadata.csv",
    index=False
)

validation_df.to_csv(
    OUTPUT_PATH / "validation_metadata.csv",
    index=False
)

test_df.to_csv(
    OUTPUT_PATH / "test_metadata.csv",
    index=False
)

print("Datasets saved successfully!\n")

print(f"Participant split table saved : {len(participant_split):,} participants")
print(f"Train metadata saved          : {len(train_df):,} records")
print(f"Validation metadata saved     : {len(validation_df):,} records")
print(f"Test metadata saved           : {len(test_df):,} records")

Datasets saved successfully!

Participant split table saved : 469 participants
Train metadata saved          : 3,608 records
Validation metadata saved     : 770 records
Test metadata saved           : 781 records


## Conclusion

A participant-level stratified data splitting strategy was successfully implemented using `train_test_split()` with stratification based on participant labels. The 469 participants were divided into training (328 participants; 70%), validation (70 participants; 15%), and test (71 participants; 15%) datasets while maintaining similar diagnostic group distributions across the three splits.

Verification confirmed that there was no overlap of participant IDs between the training, validation, and test datasets, preventing participant-level data leakage. A participant split table containing `participant_id`, `label`, and `split` was created and merged with the movement metadata so that each movement recording inherited the correct dataset assignment.

The final movement metadata contained 3,608 training recordings, 770 validation recordings, and 781 test recordings. These participant-level splits were saved for use in subsequent feature engineering, model development, hyperparameter tuning, and final performance evaluation.